In [ ]:
from pathlib import Path

import numpy as onp
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.gridspec import GridSpec
from matplotlib.axes import Axes

from per_sym import (
  DomainConfig, _2π, rotation, S, gaussian, bloch_wave, monatomic_dispersion,
  rotation3d, make_honeycomb_lattice, periodic_potential_from_sites, make_triangular_lattice,
  nearest_neighbor_vectors, find_origin_site, make_k_mesh, diatomic_dispersion
)
from plotting_utils import (
  PlotStyle, apply_matplotlib_style, _clean_axes, complex_to_rgb, save_path, subplots_list,
)

# Keep text as text in the SVG rather than converting everything to paths.
# mpl.rcParams["svg.fonttype"] = "none"
apply_matplotlib_style(PlotStyle(label_size=13, title_size=15, fig_title_size=18))


In [ ]:
# ---- config
OUTDIR = Path("media/figures")
OUTDIR.mkdir(parents=True, exist_ok=True)

def savefig(fig, name, **kwargs):
  if name is None:
    return None
  return save_path(fig, OUTDIR / name, dpi=300, facecolor="white", **kwargs)

def clean_axes(ax):
  _clean_axes(ax, grid=True, hide_spines=("top", "right"))
  ax.spines["left"].set_linewidth(1.3)
  ax.spines["bottom"].set_linewidth(1.3)


In [ ]:
# --- discrete ring (periodic BCs): sites n = 0,...,N-1
N = 24
a = 1.0
n = onp.arange(N)

# --- (A) generic state: a localized packet
f = gaussian(n, 5, 1.4)

# --- (B) Bloch / Fourier mode on the ring
m = 5
k = _2π * m / (N * a)
psi = bloch_wave(1, k, n * a)
# eigenvalue extracted numerically: should be exp(-i k a)
lam = onp.vdot(psi, S(psi)) / onp.vdot(psi, psi)

# --- (C) eigenvalues of S for the ring: Nth roots of unity
eigs = onp.exp(-1j * _2π * onp.arange(N) / N)

def plot():
  fig, ax = subplots_list(1, 3, figsize=(13, 3.6))

  # (A) shift moves a generic state
  ax[0].stem(n, f, basefmt=" ")
  ax[0].stem(n, S(f), basefmt=" ")
  ax[0].set(title="Generic state: amplitudes translate", xlabel="site n")

  # (B) Bloch mode: shift adds a constant phase
  ax[1].plot(n, onp.unwrap(onp.angle(psi)), marker="o")
  ax[1].plot(n, onp.unwrap(onp.angle(S(psi))), marker="o")
  ax[1].set(title=f"Bloch mode: Sψ = λψ,  λ ≈ {lam:.3f}", xlabel="site n",  ylabel="unwrapped phase")
  # (C) spectrum of the shift (finite ring)
  t = _2π * onp.linspace(0, 1, 400)
  ax[2].plot(onp.cos(t), onp.sin(t), alpha=0.4)
  ax[2].scatter(onp.real(eigs), onp.imag(eigs))
  ax[2].set_aspect("equal", adjustable="box")
  ax[2].set(title="spec(S): roots of unity on |z|=1", xlabel="Re", ylabel="Im")

  plt.tight_layout()
  plt.show()

In [ ]:
# --- lattice + continuous grid
cfg = DomainConfig()
a = cfg.a

x = onp.linspace(0, cfg.Lx,cfg.Nx, endpoint=False)
y = onp.linspace(0, cfg.Ly, cfg.Ny, endpoint=False)
X, Y = onp.meshgrid(x, y, indexing="xy")
grid = onp.stack((X, Y), axis=-1)
print(grid.shape)

# --- choose a periodic envelope u(r) with lattice period a
# (any a-periodic function works; cosines are the simplest)
cvec = onp.array([0.30, 0.25, 0.15])
# keep space dim along last axis i think is good?
M = onp.array([[1,0], [0,1], [1, 1]])
vec = onp.einsum('ij,...j->...i', M, grid)

u = 1.0 + onp.dot(onp.cos(_2π * vec / a), cvec)


# --- choose a Bloch wavevector k (not a reciprocal lattice vector)
kx = 0.65 * onp.pi / a
ky = 0.35 * onp.pi / a
k = onp.array([kx, ky])

psi = bloch_wave(u, k, grid)

# --- shift by one lattice step in +x: (T_a psi)(r) = psi(r - a ex)
# with periodic wrap for a clean picture on a finite patch
X_shift = (X - a) % cfg.Lx
shift_grid = onp.stack((X_shift, Y), axis=-1)
vec = onp.einsum('ij,...j->...i', M, shift_grid)

u_shift = 1.0 + onp.dot(onp.cos(_2π * vec / a), cvec)
psi_shift = bloch_wave(u_shift, k, shift_grid)





def plot(cfg: DomainConfig, psi, psi_shift):
  # --- overlay lattice sampling points
  Rx = onp.arange(0, cfg.Lx, a)
  Ry = onp.arange(0, cfg.Ly, a)
  RX, RY = onp.meshgrid(Rx, Ry, indexing="xy")

  # --- expected Bloch phase for this shift:
  # T_{a ex} gives factor e^{-i kx a}
  expected = onp.exp(-1j * kx * a)

  ratio = psi_shift / (psi + 1e-12)

  fig, axs = subplots_list(1, 3, figsize=(13.5, 4.2), constrained_layout=True, squeeze=False)


  extent = (0, cfg.Lx, 0, cfg.Ly)

  axs[0].imshow(complex_to_rgb(psi), origin="lower", extent=extent)
  axs[0].scatter(RX.ravel(), RY.ravel(), s=8, c="k", alpha=0.6)
  axs[0].set(xlabel="x", ylabel="y", title="ψ(r)  (hue=phase, brightness=|ψ|)")

  axs[1].imshow(complex_to_rgb(psi_shift), origin="lower", extent=extent)
  axs[1].scatter(RX.ravel(), RY.ravel(), s=8, c="k", alpha=0.6)
  axs[1].set(xlabel="x", ylabel="y", title="Tₐ ψ(r) = ψ(r − a eₓ)")

  axs[2].imshow(complex_to_rgb(ratio), origin="lower", extent=extent)
  axs[2].set(xlabel="x", ylabel="y", title="Tₐψ(r) / ψ(r)  (should be constant phase)")

  # annotate the expected eigenvalue
  phi = onp.angle(expected)
  axs[2].text(
    0.02 * cfg.Lx, 0.95 * cfg.Ly,
    f"expected: exp(-i kx a)\n= {expected.real:+.3f} {expected.imag:+.3f}i\nphase = {phi:+.3f} rad",
    color="w", fontsize=10, va="top",
    bbox=dict(boxstyle="round,pad=0.3", facecolor=(0,0,0,0.35), edgecolor="none")
  )

  for a_i in axs:
    a_i.set_aspect("equal")

  plt.show()

plot(cfg, psi, psi_shift)

In [ ]:
# --- parameters
cfg = DomainConfig(
  a = 1.0,                 # lattice spacing
  Nx_cell=12,             # domain contains Ncells unit cells
  Ny_cell=1,
  res_per_cell=2400//12, # dense continuous sampling for plotting
)
a = cfg.a
x = onp.linspace(0, cfg.Lx, cfg.Nx, endpoint=False)

# Bloch k (pick something not trivial)
k = 0.7 * onp.pi / a

# periodic envelope u(x+a)=u(x)
cvec = onp.array([0.35, 0.15])
M = onp.array([1, 2])
vec = x[:, None] * M
# unfort, we need cvec in the 2nd slot to keep batch dim on axis 0
u = 1.0 + onp.dot(onp.cos(_2π * vec / a), cvec)


psi = bloch_wave(u, k, x)

# translation by one lattice step: (T_a psi)(x) = psi(x-a)
# use periodic wrap on [0, L) to keep the plot clean
# translation by +a: (T_a psi)(x) = psi(x-a) with periodic wrap on [0,L)
x_shift = (x - a) % cfg.Lx # x_mina
vec = x_shift[:, None] * M
u_shift = 1.0 + onp.dot(onp.cos(_2π * vec / a), cvec)


psi_shift = bloch_wave(u_shift, k, x_shift)

# phase-corrected shifted wave: should equal psi(x) exactly (up to numerical noise)
psi_shift_phasefixed = onp.exp(1j * k * a) * psi_shift

# lattice sampling points
sites = onp.arange(0, cfg.Lx, a)
vec = sites[:, None] * M
u_sites = 1.0 + onp.dot(onp.cos(_2π * vec / a), cvec)

psi_sites = bloch_wave(u_sites, k, sites)


def plot(cfg: DomainConfig, psi, psi_shift):
  ratio = psi_shift / (psi + 1e-12)
  expected = onp.exp(-1j * onp.multiply(k, a))
  # --- make RGB strips (1 x Nx image)
  strip_psi = complex_to_rgb(psi)[None, :, :]
  strip_shift = complex_to_rgb(psi_shift)[None, :, :]
  strip_ratio = complex_to_rgb(ratio)[None, :, :]

  fig, ax = subplots_list(3, 1, figsize=(12, 4.2), constrained_layout=True)

  extent = (0, cfg.Lx, 0, cfg.Ly)
  ax[0].imshow(strip_psi, origin="lower", extent=extent, aspect="auto")
  ax[0].set_title("ψ(x)  (hue=phase, brightness=|ψ|)")

  ax[1].imshow(strip_shift, origin="lower", extent=extent, aspect="auto")
  ax[1].set_title("Tₐψ(x)=ψ(x-a)")

  ax[2].imshow(strip_ratio, origin="lower", extent=extent, aspect="auto")
  ax[2].set_title("Tₐψ(x) / ψ(x)  (Bloch ⇒ constant phase)")

  # overlay lattice sites as sample points
  sites = onp.arange(0, cfg.Lx, a)
  for a_i in ax:
    a_i.scatter(sites, 0.5*onp.ones_like(sites), s=18, c="k", alpha=0.7)
    a_i.set_yticks([])
    a_i.set_xlim(0, cfg.Lx)

  phi = onp.angle(expected)
  ax[2].text(
    0.01*cfg.Lx, 0.85,
    f"expected eigenvalue: exp(-i k a)\n= {expected.real:+.3f} {expected.imag:+.3f}i\nphase = {phi:+.3f} rad",
    color="w", fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", facecolor=(0,0,0,0.35), edgecolor="none")
  )

  ax[2].set_xlabel("x")
  plt.show()

  # ---- plot (real part as "amplitude")
  fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
  ax.plot(x, onp.real(psi), linewidth=2, label=r"$\Re\,\psi(x)$", color="grey")
  ax.plot(x, onp.real(psi_shift), linewidth=2, alpha=0.8, label=r"$\Re\,(T_a\psi)(x)=\Re\,\psi(x-a)$", color="gray")
  ax.plot(x, onp.real(psi_shift_phasefixed), "--", linewidth=2.2,
          label=r"$\Re\,(e^{ika}T_a\psi)(x)$", color="k") # (should coincide with $\Re\,\psi(x)$)

  # mark lattice sites
  for s in sites:
    ax.axvline(s, linewidth=0.6, alpha=0.25, color="k")

  # # show sampled values at lattice sites
  # original samples
  y = onp.real(psi_sites)
  ax.scatter(sites, y, s=40, zorder=5, label=r"$x_n$", color="k")
  ax.scatter(sites, y, s=30, zorder=5, color="white")

  # shifted samples, e.g. (Sx)_n = x_{n-1}
  sites_s = sites
  y_s = onp.roll(y, 1)  # illustrative discrete shift on samples

  ax.scatter(sites_s, y_s, s=40, marker="s", zorder=5, label=r"$(Sx)_n$", color="k")
  ax.scatter(sites_s, y_s, s=30, marker="s", zorder=5, color="white")

  # label a few n’s
  for n in range(6):
    xi, yi = sites[n], y[n]
    ax.annotate(rf"$x_{{{n}}}$", (xi, yi), xytext=(6, 8),
                textcoords="offset points", fontsize=11)

    xi2, yi2 = sites_s[n], y_s[n]
    ax.annotate(rf"$(Sx)_{{{n}}}$", (xi2, yi2), xytext=(6, -14),
                textcoords="offset points", fontsize=11)


  # ax.set(title=r"Shift/translation: for Bloch waves, $e^{ika}T_a\psi=\psi$")
  ax.set(
    xlabel="x",
    ylabel="amplitude (real part)",
    xlim=(0, cfg.Lx),
    xticks=[],
    yticks=[0],
  )
  ax.axhline(0, xmin=0, linewidth=0.6, alpha=0.25, color="k")
  ax.legend(loc="upper right")
  plt.tight_layout()
  plt.show()



plot(cfg, psi, psi_shift)

In [ ]:
ncells = 12
res = 2000
x = onp.linspace(-ncells, ncells, res)

# pick k0
k0 = 2.2
carrier = onp.cos(k0 * x)

# Slight envelope only so the oscillation is visible on the slide;
# conceptually, this is meant to evoke an extended plane-wave-like state.
weak_env = 0.92 + 0.08 * gaussian(x, sigma=7.0)
psi_free = weak_env * carrier

# ---- bound state in a confining well
res = 1400
Ω_xb = (-1.2, 1.2)
xb = onp.linspace(*Ω_xb, res)

# Infinite-square-well-like standing wave inside [-1,1]
inside = onp.abs(xb) <= 1.0
psi_bound = onp.full_like(xb, onp.nan)
# One standing-wave mode
psi_bound[inside] = 0.95 * onp.sin(onp.pi * (xb[inside] + 1) / 2)

# ---- continuous spectrum
E0 = 0.0
dE = 0.8
E_max = 8
E = onp.linspace(0, E_max, 400)


def make_free_vs_bound_spectrum_figure():
  fig = plt.figure(figsize=(13, 5.5), constrained_layout=True)
  gs = fig.add_gridspec(2, 2, height_ratios=[3, 2])

  # Top row: wavefunctions / amplitudes
  ax_free = fig.add_subplot(gs[0, 0])
  ax_bound = fig.add_subplot(gs[0, 1])

  # Bottom row: spectra
  ax_free_spec = fig.add_subplot(gs[1, 0])
  ax_bound_spec = fig.add_subplot(gs[1, 1])

  # --------------------------
  # Left panel: free-space state
  # --------------------------
  ax_free.plot(x, psi_free, lw=2)
  ax_free.axhline(0, lw=1, color='black', alpha=0.5)

  # Add subtle arrows to suggest translational freedom
  for x0 in [-8, -3, 2, 7]:
    ax_free.annotate(
      "",
      xy=(x0 + 1.0, 1.15),
      xytext=(x0, 1.15),
      arrowprops=dict(arrowstyle="->", lw=1.2)
    )

  ax_free.text(
    0.02, 0.95,
    "Free space:\nextended, plane-wave-like mode",
    transform=ax_free.transAxes,
    ha="left", va="top",
    fontsize=12
  )
  ax_free.text(
    0.02, 0.08,
    r"$\psi_k(x)\sim e^{ikx}$" "\n"
    r"translation symmetry  $\Rightarrow$  continuous label $k$",
    transform=ax_free.transAxes,
    ha="left", va="bottom",
    fontsize=11
  )

  ax_free.set(
    xlim=(x.min(), x.max()),
    ylim=(-1.45, 1.45),
    xticks=[],
    yticks=[],
    title="Continuous-spectrum regime",
  )
  ax_free.title.set_fontsize(14)

  # --------------------------
  # Right panel: bound state in a confining well
  # --------------------------

  # Draw confining walls
  ax_bound.plot([-1, -1], [-1.35, 1.35], lw=2.5, color='black')
  ax_bound.plot([1, 1], [-1.35, 1.35], lw=2.5, color='black')
  ax_bound.plot(xb, psi_bound, lw=2)

  # Light shading for forbidden/outside region
  ax_bound.axvspan(Ω_xb[0], -1.0, alpha=0.12)
  ax_bound.axvspan(1.0, Ω_xb[1], alpha=0.12)
  ax_bound.axhline(0, lw=1, color='black', alpha=0.5)

  ax_bound.text(
    0.02, 0.95,
    "Bound system:\nconfined standing mode",
    transform=ax_bound.transAxes,
    ha="left", va="top",
    fontsize=12
  )
  ax_bound.text(
    0.02, 0.08,
    "localization / boundary conditions\n"
    r"$\Rightarrow$ discrete normal modes",
    transform=ax_bound.transAxes,
    ha="left", va="bottom",
    fontsize=11
  )

  ax_bound.set(
    xlim=Ω_xb,
    ylim=(-1.45, 1.45),
    xticks=[],
    yticks=[],
    title="Point-spectrum regime",
  )
  ax_bound.title.set_fontsize(14)

  # --------------------------
  # Bottom-left: continuous spectrum
  # --------------------------
  ax_free_spec.plot(onp.zeros_like(E), E, alpha=0)  # set axes scale cleanly

  # Thick vertical "continuum" bar
  ax_free_spec.fill_betweenx(E, -0.18, 0.18, alpha=0.35)

  # A few labels to suggest continuous energy values
  for Ei in [1.2, 3.0, 5.5, 7.1]:
    ax_free_spec.plot([-0.22, 0.22], [Ei, Ei], lw=1.0, alpha=0.6)

  ax_free_spec.text(0.5, 0.92,
    "continuous energy continuum",
    transform=ax_free_spec.transAxes, ha="center", va="top", fontsize=11
  )
  ax_free_spec.text(0.5, 0.08,
    r"$E=\hbar^2 k^2 / 2m$" "\n" r"$k$ varies continuously",
    transform=ax_free_spec.transAxes, ha="center", va="bottom", fontsize=11
  )
  ax_free_spec.set(xlim=(-0.8, 1.0), ylim=(0, E_max), xticks=[], yticks=[], ylabel="Energy")
  ax_free_spec.yaxis.label.set_size(12)

  # --------------------------
  # Bottom-right: discrete spectrum
  # --------------------------
  discrete_levels = [0.8, 2.2, 4.4, 7.0]
  for n, En in enumerate(discrete_levels, start=1):
    ax_bound_spec.hlines(En, -0.45, 0.45, lw=2.2)
    ax_bound_spec.text(0.52, En, rf"$E_{n}$", va="center", fontsize=11)

  ax_bound_spec.text(0.5, 0.92,
    "discrete bound-state levels",
    transform=ax_bound_spec.transAxes, ha="center", va="top", fontsize=11)
  ax_bound_spec.text(0.5, 0.08,
    "normalizable standing states",
    transform=ax_bound_spec.transAxes, ha="center", va="bottom", fontsize=11)
  ax_bound_spec.set(xlim=(-0.8, 1.0), ylim=(0, E_max), xticks=[], yticks=[])


  fig.suptitle(
    "Extended free-space modes vs confined bound states:\n"
    "continuous spectrum and discrete spectrum",
    fontsize=16
  )
  return fig


if __name__ == "__main__":
  fig = make_free_vs_bound_spectrum_figure()
  # savefig(fig, "free_vs_bound_spectrum.png", bbox_inches="tight")
  plt.show()

In [ ]:
cfg = DomainConfig(
  a=1.0
)
n_cells = 8
a = cfg.a
sigma = 0.18

# Lattice sites
x_sites = onp.arange(n_cells) * a
x_min = -0.6 * a
x_max = (n_cells - 1 + 0.6) * a
x = onp.linspace(x_min, x_max, 3000)


# Zone-boundary standing waves: cos(pi x / a) and sin(pi x / a)
# Shift sin by a/2 so its maxima sit between lattice sites.
psi_lo = onp.cos(_2π * x / (2 * a))
psi_hi = onp.sin(_2π * x / (2 * a))

# Periodic potential: wells at lattice sites
V = onp.zeros_like(x)
for xs in x_sites:
  V -= 1.0 * gaussian(x, xs, sigma)

# Normalize / rescale potential for plotting
V = V / onp.max(onp.abs(V))
V_plot = 0.55 * V - 0.9


# ---- energy splitting
E0 = 0.0
dE = 0.8
Emin = E0 - dE / 2
Emax = E0 + dE / 2

def make_zone_edge_splitting_figure():

  # Set up figure
  fig = plt.figure(figsize=(13.5, 7.5), constrained_layout=True)
  gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1.0], height_ratios=[1, 1])

  ax1 = fig.add_subplot(gs[0, 0])
  ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
  axE = fig.add_subplot(gs[:, 1])

  # ---------- Top panel: low-energy standing wave ----------
  ax1.plot(x, V_plot, lw=2.0, color="0.35")
  ax1.plot(x, psi_lo, lw=2.5)

  # Lattice sites
  for xs in x_sites:
    ax1.axvline(xs, ymin=0.06, ymax=0.94, lw=0.8, color="0.8", zorder=0)
    ax1.add_patch(Circle((xs, -1.25), 0.045, fill=True, color="0.15"))

  # Mark maxima near low-potential sites
  site_vals = onp.cos(onp.pi * x_sites / a)
  ax1.scatter(x_sites, site_vals, s=45, zorder=5)

  ax1.text(0.02, 0.93,
    r"Lower-energy combination  $\psi_{-}(x)\propto \cos(\pi x/a)$",
    transform=ax1.transAxes, ha="left", va="top", fontsize=14)
  ax1.text(0.02, 0.82,
    "Antinodes centered on lattice wells\nmore weight in low-potential regions",
    transform=ax1.transAxes, ha="left", va="top")
  ax1.set(ylim=(-1.45, 1.25), xticks=[], yticks=[], title="Zone-edge standing-wave splitting in a periodic potential")

  # ---------- Bottom panel: high-energy standing wave ----------
  ax2.plot(x, V_plot, lw=2.0, color="0.35")
  ax2.plot(x, psi_hi, lw=2.5)

  # Lattice sites
  for xs in x_sites:
    ax2.axvline(xs, ymin=0.06, ymax=0.94, lw=0.8, color="0.8", zorder=0)
    ax2.add_patch(Circle((xs, -1.25), 0.045, fill=True, color="0.15"))

  # Mark maxima between lattice sites
  x_mid = x_sites[:-1] + 0.5 * a
  mid_vals = onp.sin(onp.pi * x_mid / (2 * a))
  ax2.scatter(x_mid, mid_vals, s=45, zorder=5)

  ax2.text(
    0.02, 0.93,
    r"Higher-energy combination  $\psi_{+}(x)\propto \sin(\pi x/a)$",
    transform=ax2.transAxes,
    ha="left", va="top",
    fontsize=14
  )
  ax2.text(
    0.02, 0.82,
    "Antinodes shifted toward high-potential regions\nless weight on the lattice wells",
    transform=ax2.transAxes,
    ha="left", va="top"
  )
  ax2.set(ylim=(-1.45, 1.25), yticks=[], xlabel="Position along lattice")
  ax2.set_ylabel(ylabel="Mode amplitude", labelpad=18)

  # ---------- Right panel: energy splitting schematic ----------
  
  axE.hlines(E0, 0.18, 0.82, lw=2.2, linestyles="dashed", color="0.55")
  axE.hlines(Emin, 0.18, 0.82, lw=3.0)
  axE.hlines(Emax, 0.18, 0.82, lw=3.0)

  axE.text(0.86, E0, r"degenerate free-wave level", va="center", fontsize=13, color="0.4")
  axE.text(0.86, Emin, r"$E_{-}$", va="center", fontsize=14)
  axE.text(0.86, Emax, r"$E_{+}$", va="center", fontsize=14)

  axE.annotate(
    "",
    xy=(0.58, Emax),
    xytext=(0.58, Emin),
    arrowprops=dict(arrowstyle="<->", lw=1.8)
  )
  axE.text(0.61, 0.5 * (Emin + Emax), "gap", va="center", fontsize=14)

  axE.text(
    0.04, 0.95,
    "At the Brillouin-zone edge,\n"
    r"$k=\pi/a$, the free states are degenerate." "\n"
    "The periodic potential selects\n"
    "symmetry-adapted standing waves\n"
    "with different energies.",
    transform=axE.transAxes,
    ha="left", va="top",
    fontsize=14
  )

  axE.set(
    xlim=(0, 1.55), ylim=(-1.2, 1.2), xticks=[], yticks=[],
    title="Gap opening",
  )
  for spine in axE.spines.values():
    spine.set_visible(False)

  # ---------- Cosmetic cleanup ----------
  for ax in (ax1, ax2):
    for spine in ["top", "right", "left"]:
      ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_alpha(0.4)

  return fig


if __name__ == "__main__":
  fig = make_zone_edge_splitting_figure()
  # savefig(fig, "zone_edge_standing_wave_splitting.png", bbox_inches="tight")
  plt.show()

In [ ]:


def plot_honeycomb_and_potential(
  n1=4,
  n2=4,
  a=1.0,
  sigma=0.20,
  depth=1.0,
  grid_pts=500,
  cmap="viridis",
  savepath=None,
):
  ptsA, ptsB, A1, A2, bA, bB = make_honeycomb_lattice(n1, n2, a=a)
  pts = onp.vstack([ptsA, ptsB])

  xmin = pts[:, 0].min() - 1.0 * a
  xmax = pts[:, 0].max() + 1.0 * a
  ymin = pts[:, 1].min() - 1.0 * a
  ymax = pts[:, 1].max() + 1.0 * a

  x = onp.linspace(xmin, xmax, grid_pts)
  y = onp.linspace(ymin, ymax, grid_pts)
  X, Y = onp.meshgrid(x, y)

  V = periodic_potential_from_sites(X, Y, pts, sigma=sigma, depth=depth)

  fig, ax = plt.subplots(figsize=(8.5, 8), constrained_layout=True)

  im = ax.imshow(
    V,
    extent=(xmin, xmax, ymin, ymax),
    origin="lower",
    cmap=cmap,
    aspect="equal",
  )

  ax.scatter(ptsA[:, 0], ptsA[:, 1], s=18, c="red", edgecolors="black", linewidths=0.5, label="A")
  ax.scatter(ptsB[:, 0], ptsB[:, 1], s=18, c="white", edgecolors="black", linewidths=0.5, label="B")

  ax.set(
    title="Honeycomb lattice = triangular Bravais lattice + 2-point basis",
    xlabel="x",
    ylabel="y",
    aspect="equal",
  )
  ax.legend(loc="upper right")

  cbar = fig.colorbar(im, ax=ax, shrink=0.85)
  cbar.set_label(r"$V(x,y)$")

  savefig(fig, savepath, bbox_inches="tight")
  return fig, ax




def plot_triangular_lattice_and_potential(
  n1=4,
  n2=4,
  a=1.0,
  sigma=0.23,
  depth=1.0,
  grid_pts=500,
  show_neighbors=True,
  show_primitive_vectors=True,
  cmap="viridis",
  savepath=None,
):
  pts, ij, a1, a2 = make_triangular_lattice(n1, n2, a=a)

  # Plot window
  xmin = pts[:, 0].min() - 0.8 * a
  xmax = pts[:, 0].max() + 0.8 * a
  ymin = pts[:, 1].min() - 0.8 * a
  ymax = pts[:, 1].max() + 0.8 * a

  x = onp.linspace(xmin, xmax, grid_pts)
  y = onp.linspace(ymin, ymax, grid_pts)
  X, Y = onp.meshgrid(x, y)

  V = periodic_potential_from_sites(X, Y, pts, sigma=sigma, depth=depth)

  fig, ax = plt.subplots(figsize=(8.5, 8), constrained_layout=True)

  im = ax.imshow(
    V,
    extent=(xmin, xmax, ymin, ymax),
    origin="lower",
    cmap=cmap,
    aspect="equal",
  )

  # Lattice sites
  ax.scatter(pts[:, 0], pts[:, 1], s=18, c="white", edgecolors="black", linewidths=0.5, zorder=3)

  origin_idx = find_origin_site(pts)
  r0 = pts[origin_idx]

  # Highlight one site
  ax.scatter([r0[0]], [r0[1]], s=55, c="red", zorder=4)

  if show_neighbors:
    nn = nearest_neighbor_vectors(a1, a2)
    for d in nn:
      r = r0 + d
      ax.plot([r0[0], r[0]], [r0[1], r[1]], "--",lw=2.0, color="black", alpha=0.8, zorder=2)
      ax.scatter([r[0]], [r[1]], s=45, c="red", edgecolors="black", linewidths=0.5, zorder=4)

  if show_primitive_vectors:
    ax.arrow(
      r0[0], r0[1], a1[0], a1[1],
      width=0.025 * a, head_width=0.16 * a, head_length=0.18 * a,
      length_includes_head=True, color="cyan", zorder=5
    )
    ax.arrow(
      r0[0], r0[1], a2[0], a2[1],
      width=0.025 * a, head_width=0.16 * a, head_length=0.18 * a,
      length_includes_head=True, color="magenta", zorder=5
    )
    ax.text(*(r0 + 1.08 * a1), r"$a_1$", color="cyan", fontsize=13)
    ax.text(*(r0 + 1.08 * a2), r"$a_2$", color="magenta", fontsize=13)

  ax.set(title="Triangular (hexagonal) Bravais lattice with periodic potential", 
        xlabel="x",ylabel="y", aspect="equal")

  cbar = fig.colorbar(im, ax=ax, shrink=0.85)
  cbar.set_label(r"$V(x,y)$")

  if savepath is not None:
    fig.savefig(savepath, dpi=300, bbox_inches="tight")

  return fig, ax


if __name__ == "__main__":
  plot_triangular_lattice_and_potential(
    n1=4,
    n2=4,
    a=1.0,
    sigma=0.23,
    depth=1.0,
    grid_pts=500,
    show_neighbors=True,
    show_primitive_vectors=True,
    cmap="magma_r",
    savepath="triangular_lattice_potential.png",
  )
  plt.show()

  plot_honeycomb_and_potential(savepath="honeycomb_potential.png", cmap="magma_r")
  plt.show()

In [ ]:
kmax = 2.3
ngrid = 220
KX, KY = make_k_mesh(kmax, ngrid)
E = 0.5 * (KX**2 + KY**2)

# Highlight one equal-energy contour
E0 = 1.15
kr = onp.sqrt(2 * E0)
theta = _2π * onp.linspace(0, 1, 400)
xring = kr * onp.cos(theta)
yring = kr * onp.sin(theta)
zring = onp.full_like(theta, E0 + 0.02)

# ---- central attractive potential
r = onp.linspace(0.18, 8.0, 1400)
V = -1.65 / onp.sqrt(r**2 + 0.16)


 # Bound-state energies (schematic)
E_bound = onp.array([-1.00, -0.43, -0.21, -0.12, -0.075])
Elabels = [r"$1s$", r"$2s/2p$", r"$3\ell$", r"$4\ell$", r"$5\ell$"]


# Reference degenerate level
Edeg = 2.4
Eax1 = 2.58
Eax2 = 2.22
Elow = [2.68, 2.38, 2.08]



def make_symmetry_breaking_spectrum_figure():
  fig = plt.figure(figsize=(15.5, 6.2), constrained_layout=True)
  gs = GridSpec(1, 3, figure=fig, width_ratios=[1.15, 1.0, 1.05])

  # ============================================================
  # Panel 1: free particle
  # ============================================================
  ax1 = fig.add_subplot(gs[0, 0], projection="3d")

  ax1.plot_surface(KX, KY, E,
    rstride=4, cstride=4, linewidth=0,
    antialiased=True, alpha=0.9)

  
  ax1.plot(xring, yring, zring, lw=3)

  ax1.set(title="Free particle", xlabel=r"$k_x$", ylabel=r"$k_y$", zlabel=r"$E$")
  ax1.text2D(0.03, 0.96, 
    r"$E=\frac{\hbar^2}{2m}(k_x^2+k_y^2)$" "\n" "continuous momentum labels",
    transform=ax1.transAxes, va="top")
  ax1.text2D(0.03, 0.08,
    "full translation + rotation symmetry",
    transform=ax1.transAxes, va="bottom")
  ax1.view_init(elev=27, azim=-55)

  # ============================================================
  # Panel 2: central attractive potential
  # ============================================================
  ax2 = fig.add_subplot(gs[0, 1])

  

  ax2.plot(r, V, lw=2.5)

 

  for Eb, lab in zip(E_bound, Elabels):
    idx = onp.argmax(V > Eb)
    x_right = r[idx] if idx > 0 else 1.2
    ax2.hlines(Eb, 0.18, x_right, lw=2.3)
    ax2.text(x_right + 0.12, Eb, lab, va="center", fontsize=12)

  # Continuum threshold
  ax2.axhline(0.0, lw=1.7, ls="--")
  ax2.axhspan(0.0, 0.7, alpha=0.12)

  ax2.text(
    0.60, 0.92,
    "scattering continuum\n$E>0$",
    transform=ax2.transAxes,
    ha="left", va="top"
  )
  ax2.text(
    0.60, 0.14,
    "discrete bound levels\n$E<0$",
    transform=ax2.transAxes,
    ha="left", va="bottom"
  )
  ax2.text(
    0.03, 0.96,
    "translation broken,\nrotation retained",
    transform=ax2.transAxes,
    va="top"
  )

  ax2.set(title="Central attractive potential", xlabel=r"$r$", ylabel="Energy", xlim=(0.18, 8.0), ylim=(-1.85, 0.70))


  for spine in ["top", "right"]:
    ax2.spines[spine].set_visible(False)

  # ============================================================
  # Panel 3: explicit degeneracy splitting under symmetry lowering
  # ============================================================
  ax3 = fig.add_subplot(gs[0, 2])

  ax3.set(title="Lower symmetry splits levels", xlim=(-0.2, 3.2), ylim=(-0.1, 3.2), 
          xticks=[0.3, 1.6, 2.8], xticklabels=["spherical", "axial", "low symmetry"], ylabel="Energy")



  # Spherical: 3-fold degenerate
  ax3.hlines(Edeg, 0.0, 0.6, lw=2.8, colors="k")
  for x in [0.10, 0.30, 0.50]:
    ax3.plot(x, Edeg, marker="o", ms=5, c="k")
  ax3.text(0.30, Edeg + 0.12, "3-fold", ha="center")

  from matplotlib.axes import Axes
  levels = lambda ax, E: Axes.hlines(ax, E, 1.25, 1.95, lw=2.8, colors="k")
  # Axial: doublet + singlet
  levels(ax3, Eax1)
  levels(ax3, Eax2)

  for x in [1.42, 1.78]:
    ax3.plot(x, Eax1, marker="o", ms=5, c="k")
  ax3.plot(1.60, Eax2, marker="o", ms=5, c="k")
  ax3.text(1.60, Eax1 + 0.12, "2-fold", ha="center")
  ax3.text(1.60, Eax2 - 0.20, "1-fold", ha="center")

  # Low symmetry: all split
  for i, E_i in enumerate(Elow):
    ax3.hlines(E_i, 2.45, 3.05, lw=2.8, colors="k")
    ax3.plot(2.75, E_i, marker="o", ms=5, c="k")
    ax3.text(3.09, E_i, rf"$E_{i+1}$", va="center")

  # Arrows showing symmetry reduction
  ax3.annotate("", xy=(1.25, Eax1), xytext=(0.60, Edeg),
    arrowprops=dict(arrowstyle="->", lw=1.6))
  ax3.annotate("", xy=(1.25, Eax2), xytext=(0.60, Edeg),
    arrowprops=dict(arrowstyle="->", lw=1.6))

  for E_i in Elow:
    ax3.annotate("", xy=(2.45, E_i), xytext=(1.95, 0.5 * (Eax1 + Eax2)),
      arrowprops=dict(arrowstyle="->", lw=1.4))

  ax3.text(0.03, 0.96,
    "same operator family,\nless symmetry,\nsmaller degeneracies",
    transform=ax3.transAxes, va="top")

  for spine in ["top", "right"]:
    ax3.spines[spine].set_visible(False)

  fig.suptitle("Symmetry reduction reorganizes the spectrum", y=1.02)
  return fig


if __name__ == "__main__":
  fig = make_symmetry_breaking_spectrum_figure()
  # savefig(fig, "symmetry_breaking_reorganizes_spectrum.png", bbox_inches="tight")
  plt.show()

In [ ]:
# ----------------------------
# Global presentation styling
# ----------------------------
apply_matplotlib_style(PlotStyle(
  figure_dpi=180,
  label_size=18,
  title_size=22,
  tick_size=15,
  legend_size=15,
  fig_title_size=18,
  font_family="serif",
  lw_main=3.0,
))
plt.rcParams.update({
  "figure.figsize": (10, 6),
  "axes.spines.top": False,
  "axes.spines.right": False,
  "savefig.bbox": "tight",
})


def style_band_axes(ax, a=1.0):
  kmin = -onp.pi / a
  kmax = onp.pi / a

  ax.set_xlim(kmin, kmax)
  ax.set_xticks([-onp.pi / a, 0.0, onp.pi / a])
  ax.set_xticklabels([r"$-\pi/a$", r"$0$", r"$\pi/a$"], fontsize=13)

  ax.set_xlabel(r"$k$", fontsize=15)
  ax.set_ylabel(r"$\omega$", fontsize=15)

  ax.spines["top"].set_visible(False)
  ax.spines["right"].set_visible(False)

  ax.axvline(-onp.pi / a, linestyle="--", linewidth=1.0, alpha=0.35)
  ax.axvline(0.0, linestyle="--", linewidth=1.0, alpha=0.35)
  ax.axvline(onp.pi / a, linestyle="--", linewidth=1.0, alpha=0.35)

  ax.tick_params(axis="both", labelsize=12)
  ax.grid(False)


In [ ]:

# 1) Monatomic vs diatomic 1D chain dispersion
a = 1.0
K = 1.0

# Monatomic parameters
M_mono = 1.0

# Diatomic parameters
m1 = 1.0
m2 = 2.4

k = onp.linspace(-onp.pi / a, onp.pi / a, 1200)

# Monatomic branch
ω_mono = monatomic_dispersion(k, K, M_mono, a)

# Diatomic branches
ω_acoustic, ω_optical = diatomic_dispersion(k, K, m1, m2)


# 2) Folded parabola + gap opening

G = _2π / a
kb = onp.pi / a
gap = 0.8

k_full = 3.2 * onp.pi * onp.linspace(-1, 1, 3000)

# Free-electron parabola and shifted copies
E0 = k_full ** 2
Em = (k_full - G) ** 2
Ep = (k_full + G) ** 2

# Fold into first BZ
k_bz = onp.linspace(-kb, kb, 1200)
e1 = k_bz ** 2
e2 = (k_bz - G) ** 2
e3 = (k_bz + G) ** 2

# Sort pointwise to get the lowest folded branches
stacked = onp.vstack([e1, e2, e3])
folded = onp.sort(stacked, axis=0)

# Open a gap by coupling the two lowest nearly crossing branches
# This is a schematic 2x2 avoided-crossing model near the boundary.
d1 = k_bz ** 2
d2 = (k_bz - G) ** 2
avg = 0.5 * (d1 + d2)
diff = 0.5 * (d1 - d2)
lower = avg - onp.sqrt(diff ** 2 + (gap / 2.0) ** 2)
upper = avg + onp.sqrt(diff ** 2 + (gap / 2.0) ** 2)



# 3) Simple crystal-like band structure with a visible gap

x = onp.linspace(0.0, 1.0, 1000)

# Simple stylized bands
band1 = 0.35 + 1.4 * onp.sin(0.5 * onp.pi * x) ** 2
band2 = 2.7 + 0.8 * onp.sin(onp.pi * x) ** 2
band3 = 4.6 + 0.9 * onp.sin(0.7 * onp.pi * x + 0.25) ** 2

# ============================================================
# 1) Monatomic vs diatomic 1D chain dispersion
# ============================================================
def plot_mono_vs_diatomic():
  """
  Monatomic chain:
    ω(k) = 2 sqrt(K/M) |sin(ka/2)|

  Diatomic chain:
    ω^2 = K(1/m + 1/M) ± K sqrt((1/m + 1/M)^2 - 4/(mM) sin^2(ka/2))
  """



  fig, axes = subplots_list(1, 2, figsize=(14, 5.5), sharey=True)
  # -------- left: monatomic
  ax = axes[0]
  ax.plot(k, ω_mono)
  ax.set(title="Monatomic chain", xlabel=r"$k$", ylabel=r"$\omega(k)$", xlim=(-onp.pi / a, onp.pi / a),
          xticks=[-onp.pi, 0, onp.pi], xticklabels=[r"$-\pi/a$", r"$0$", r"$\pi/a$"])

  ax.annotate("one degree of freedom\nper unit cell",
              xy=(1.6, 1.2),
              xytext=(0.5, 1.8),
              arrowprops=dict(arrowstyle="->", lw=1.5))
  clean_axes(ax)

  # -------- right: diatomic
  ax = axes[1]
  ax.plot(k, ω_acoustic, label="acoustic")
  ax.plot(k, ω_optical, label="optical")
  ax.set(title="Diatomic chain", xlabel=r"$k$",  xlim=(-onp.pi / a, onp.pi / a),
          xticks=[-onp.pi, 0, onp.pi], xticklabels=[r"$-\pi/a$", r"$0$", r"$\pi/a$"])

  # gap marker at zone boundary
  kb = onp.pi / a
  idx = onp.argmin(onp.abs(k - kb))
  gap_low = ω_acoustic[idx]
  gap_high = ω_optical[idx]
  ax.vlines(kb * 0.92, gap_low, gap_high, linestyles="dashed")
  ax.text(kb * 0.53, 0.5 * (gap_low + gap_high), "gap", va="center")

  ax.annotate("two degrees of freedom\nper unit cell",
              xy=(0.0, ω_optical[len(k)//2]),
              xytext=(-2.4, 2.4),
              arrowprops=dict(arrowstyle="->", lw=1.5))
  ax.legend(frameon=False, loc="upper center")
  clean_axes(ax)

  fig.suptitle("More structure per cell changes the spectrum", y=1.03)
  return fig


# ============================================================
# 2) Folded parabola + gap opening
# ============================================================


def plot_folded_parabola_gap():
  """
  Schematic nearly-free-electron picture in 1D:
  Start with E(k)=k^2 and its shifted copies (k-G)^2, (k+G)^2.
  Fold them into the first Brillouin zone and then show a gap opening
  near the zone boundary by coupling two branches.
  """
  fig, axes = subplots_list(1, 3, figsize=(18, 5.5), sharey=True)

  # -------- panel 1: free parabola copies
  ax = axes[0]
  ax.plot(k_full, E0)
  ax.plot(k_full, Em, alpha=0.6)
  ax.plot(k_full, Ep, alpha=0.6)
  ax.axvline(-kb, ls="--", lw=1.5)
  ax.axvline(kb, ls="--", lw=1.5)
  ax.set(
    xlim=(-3.2 * onp.pi, 3.2 * onp.pi), ylim=(0, 20), title="Free-electron parabola", 
    xlabel=r"$k$", ylabel=r"$E(k)$", xticks=[-2*onp.pi, -onp.pi, 0, onp.pi, 2*onp.pi],
    xticklabels=[r"$-2\pi/a$", r"$-\pi/a$", r"$0$", r"$\pi/a$", r"$2\pi/a$"],
  )
  clean_axes(ax)

  # -------- panel 2: folded bands
  ax = axes[1]
  ax.plot(k_bz, folded[0])
  ax.plot(k_bz, folded[1])
  ax.plot(k_bz, folded[2], alpha=0.65)
  ax.axvline(-kb, ls="--", lw=1.5)
  ax.axvline(kb, ls="--", lw=1.5)
  ax.set(xlim=(-kb, kb), title="Fold into first Brillouin zone", xlabel=r"$k$", xticks=[-kb, 0, kb], 
         xticklabels=[r"$-\pi/a$", r"$0$", r"$\pi/a$"])
  clean_axes(ax)

  # -------- panel 3: avoided crossing / gap
  ax = axes[2]
  ax.plot(k_bz, lower, label="lower band")
  ax.plot(k_bz, upper, label="upper band")
  ax.axvline(-kb, ls="--", lw=1.5)
  ax.axvline(kb, ls="--", lw=1.5)
  ax.set(xlim=(-kb, kb), title="Periodicity opens a gap", xlabel=r"$k$", xticks=[-kb, 0, kb], 
          xticklabels=[r"$-\pi/a$", r"$0$", r"$\pi/a$"])
  i = onp.argmin(onp.abs(k_bz - kb))
  ax.vlines(0.92 * kb, lower[i], upper[i], linestyles="dashed")
  ax.text(0.53 * kb, 0.5 * (lower[i] + upper[i]), "gap", va="center")
  ax.legend(frameon=False, loc="upper center")
  clean_axes(ax)

  fig.suptitle("From free-electron dispersion to a zone-edge gap", y=1.03)
  return fig



# ============================================================
# 3) Simple crystal-like band structure with a visible gap
# ============================================================
def plot_simple_band_structure():
  """
  A stylized 1D crystal band structure over the path Γ -> X.
  This is deliberately schematic, not material-specific.
  """
  # Create a visible gap between band1 and band2
  fig, ax = plt.subplots(figsize=(8.5, 6))
  ax.plot(x, band1)
  ax.plot(x, band2)
  ax.plot(x, band3)

  # Gap shading
  gap_bottom = band1.max()
  gap_top = band2.min()
  ax.axhspan(gap_bottom, gap_top, alpha=0.12)
  ax.text(0.76, 0.5 * (gap_bottom + gap_top), "band gap", va="center")

  ax.set(xlim=(0, 1), ylim=(0, 6.2), title="Stylized crystal band structure",
        ylabel="Energy", xticks=[0, 1], xticklabels=[r"$\Gamma$", r"$X$"])
  ax.annotate("valence band", xy=(0.22, band1[220]), xytext=(0.06, 1.8),
              arrowprops=dict(arrowstyle="->", lw=1.5))
  ax.annotate("conduction band", xy=(0.52, band2[520]), xytext=(0.11, 3.85),
              arrowprops=dict(arrowstyle="->", lw=1.5))
  clean_axes(ax)
  return fig


if __name__ == "__main__":
  fig = plot_mono_vs_diatomic()
  # savefig(fig, "mono_vs_diatomic_dispersion.png")
  plt.show()
  fig = plot_folded_parabola_gap()
  # savefig(fig, "folded_parabola_gap.png")
  plt.show()
  fig = plot_simple_band_structure()
  # savefig(fig, "simple_crystal_band_structure.png")
  plt.show()



In [ ]:

# ---------------------------
# Geometry
# ---------------------------
k = onp.array([1.2, 0.8, 1.5])
k_norm = onp.linalg.norm(k)

axis = onp.array([0.0, 0.0, 1.0])
theta = onp.deg2rad(55.0)

R = rotation3d(axis, theta)
k_rot = R @ k

# Sphere |k| = const
u = onp.linspace(0.0, 2.0 * onp.pi, 180)
v = onp.linspace(0.0, onp.pi, 90)

x = k_norm * onp.outer(onp.cos(u), onp.sin(v))
y = k_norm * onp.outer(onp.sin(u), onp.sin(v))
z = k_norm * onp.outer(onp.ones_like(u), onp.cos(v))

# ---------------------------
# Figure
# ---------------------------
fig = plt.figure(figsize=(7.2, 7.2), dpi=180)
ax = fig.add_subplot(111, projection="3d")
ax.set_proj_type("ortho")

# Transparent background
fig.patch.set_alpha(0.0)
ax.set_facecolor((1, 1, 1, 0))

# Subtle sphere
ax.plot_surface(
  x, y, z,
  rstride=3,
  cstride=3,
  color="#d9d9d9",
  alpha=0.12,
  edgecolor="none",
  shade=False
)

# Optional faint great circle in the z=0 plane
t = onp.linspace(0.0, 2.0 * onp.pi, 400)
ax.plot(k_norm * onp.cos(t), k_norm * onp.sin(t),
  0.0 * t,
  linewidth=1.0,
  alpha=0.25,
  color="black"
)

# Vectors
ax.quiver(
  0, 0, 0,
  k[0], k[1], k[2],
  arrow_length_ratio=0.08,
  linewidth=2.2,
  color="black"
)

ax.quiver(
  0, 0, 0,
  k_rot[0], k_rot[1], k_rot[2],
  arrow_length_ratio=0.08,
  linewidth=2.2,
  color="#666666"
)

# Endpoints
ax.scatter([k[0], k_rot[0]], [k[1], k_rot[1]], [k[2], k_rot[2]],
  s=26, color=["black", "#666666"], depthshade=False)

# Labels
label_offset = 0.08 * k_norm
ax.text(
  k[0] + label_offset,
  k[1] + label_offset,
  k[2] + label_offset,
  r"$\mathbf{k}$",
  fontsize=16
)

ax.text(
  k_rot[0] + label_offset,
  k_rot[1] + label_offset,
  k_rot[2] + label_offset,
  r"$R\mathbf{k}$",
  fontsize=16
)

# Clean framing
lim = 1.18 * k_norm
ax.set_box_aspect((1, 1, 1))

# Remove all axis furniture
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), zlim=(-lim, lim), xticks=[], yticks=[], zticks=[])
ax.grid(False)
ax.set_axis_off()

# View angle
ax.view_init(elev=22, azim=36)

# Figure-level annotation instead of axis title
fig.text(
  0.5, 0.93,
  r"$E(\mathbf{k}) = E(R\mathbf{k})$",
  ha="center",
  va="center",
  fontsize=18
)

plt.tight_layout(pad=0.2)
# savefig(fig, "fs_k_sphere.png")
plt.show()

In [ ]:
import numpy as onp
import matplotlib.pyplot as plt

def coulomb_like(r, depth=1.0, soft=0.18):
  return -depth / (r + soft)

def scattering_mode(x, k, amp=0.11):
  return amp * onp.sin(k * x)

def bound_mode(x, n, amp=0.16, decay=2.2):
  # qualitative only: oscillatory near the origin, decaying at large r
  return amp * (x ** 0.7) * onp.exp(-decay * x) * onp.sin((n + 1) * onp.pi * x)

fig = plt.figure(figsize=(10.8, 4.4), dpi=220, constrained_layout=True)
gs = fig.add_gridspec(2, 2, width_ratios=[1.05, 1.35], height_ratios=[1, 1])

axV = fig.add_subplot(gs[:, 0])
axScat = fig.add_subplot(gs[0, 1])
axBound = fig.add_subplot(gs[1, 1])

# -------------------------
# Left panel: potential V(r)
# -------------------------
r = onp.linspace(0.03, 4.2, 600)
V = coulomb_like(r, depth=0.95, soft=0.16)

axV.plot(r, V, lw=2.6)
axV.axhline(0.0, ls='--', lw=1.4)

# sample bound energies
bound_E = onp.array([-0.72, -0.34, -0.18, -0.09])
for E in bound_E:
  axV.hlines(E, xmin=0.10, xmax=3.55, lw=2.0)

axV.text(3.62, 0.02, r'$E=0$', fontsize=11, va='bottom')
axV.text(2.15, -0.83, r'$V(r)$', fontsize=12)

axV.set(
  xlabel=r'$r$',
  ylabel='energy',
  xlim=(0.0, 4.2),
  ylim=(-1.05, 0.26),
  title='Central attractive potential'
)
axV.spines['top'].set_visible(False)
axV.spines['right'].set_visible(False)

# -------------------------
# Upper-right: E > 0 continuum
# -------------------------
x = onp.linspace(0.0, 1.0, 700)
ys = [0.78, 0.52, 0.26]
ks = [7.0, 10.5, 14.0]

for y0, k in zip(ys, ks):
  axScat.hlines(y0, xmin=0.0, xmax=1.0, lw=0.8, alpha=0.35)
  axScat.plot(x, y0 + scattering_mode(x, k), lw=2.0)
  axScat.text(1.02, y0, fr'$k \approx {k:.1f}$', va='center', fontsize=10)

axScat.text(0.02, 0.95, r'$E>0$', fontsize=12)
axScat.text(0.26, 0.95, 'continuum of scattering states', fontsize=11)

axScat.set(
  xlim=(0.0, 1.15),
  ylim=(0.05, 1.02),
  xticks=[],
  yticks=[],
  title='No decay at infinity  →  continuous energies'
)
for spine in axScat.spines.values():
  spine.set_visible(False)

# -------------------------
# Lower-right: E < 0 bound states
# -------------------------
bound_y = [0.78, 0.54, 0.30]
nodes = [0, 1, 2]
decays = [3.0, 2.3, 1.75]

for y0, n, d in zip(bound_y, nodes, decays):
  axBound.hlines(y0, xmin=0.0, xmax=1.0, lw=0.8, alpha=0.35)
  mode = bound_mode(x, n=n, decay=d)
  axBound.plot(x, y0 + mode, lw=2.0)
  axBound.text(1.02, y0, f'{n} radial node' + ('' if n == 1 else 's'), va='center', fontsize=10)

axBound.text(0.02, 0.95, r'$E<0$', fontsize=12)
axBound.text(0.22, 0.95, 'discrete bound modes', fontsize=11)

axBound.set(
  xlim=(0.0, 1.18),
  ylim=(0.08, 1.02),
  xticks=[],
  yticks=[],
  title='Decay at infinity  →  only special modes survive'
)
for spine in axBound.spines.values():
  spine.set_visible(False)

fig.suptitle('Hydrogen: continuum above threshold, quantized bound states below', fontsize=14)

fig.savefig('hydrogen_bound_vs_scattering.svg', transparent=True, bbox_inches='tight')
fig.savefig('hydrogen_bound_vs_scattering.png', transparent=False, bbox_inches='tight')
plt.show()